In [7]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
import joblib
import os

# Load dataset
df = pd.read_csv('/Users/jinenmodi/ImpData/supplychain-genai-optim/data/processed/cleaned_walmart_data.csv', parse_dates=['Date'])

# Preprocess
df = df[df['Weekly_Sales'] > 0].sort_values('Date').reset_index(drop=True)
df['Store'] = df['Store'].astype(int)
df['Dept'] = df['Dept'].astype(int)
df['Type'] = df['Type'].astype('category').cat.codes

# Features and raw target
features = [
    'Store', 'Dept', 'Type', 'IsHoliday',
    'Temperature', 'Fuel_Price', 'CPI', 'Unemployment',
    'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5',
    'Year', 'Month', 'Week', 'Day', 'Quarter', 'IsMonthStart', 'IsMonthEnd'
]
target = 'Weekly_Sales'

# Model save path
save_dir = '/Users/jinenmodi/ImpData/supplychain-genai-optim/models/groupwise_by_dept'
os.makedirs(save_dir, exist_ok=True)

# Evaluation metrics
def smape(y_true, y_pred):
    return 100 * np.mean(2.0 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred) + 1e-8))

# Results container
results = []

# Train one model per Dept
for dept_id in df['Dept'].unique():
    dept_df = df[df['Dept'] == dept_id].copy()
    dept_df = dept_df.sort_values('Date')

    if len(dept_df) < 2000:
        continue

    X = dept_df[features]
    y = dept_df[target]

    # 80/20 time-aware split
    split_idx = int(len(X) * 0.8)
    X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
    y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

    model = xgb.XGBRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        tree_method='hist',
        objective='reg:squarederror'
    )

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    smape_val = smape(y_test, y_pred)

    results.append({
        'Dept': dept_id,
        'Train Size': len(X_train),
        'Test Size': len(X_test),
        'RMSE': round(rmse, 2),
        'MAE': round(mae, 2),
        'SMAPE': round(smape_val, 2)
    })

    model_filename = os.path.join(save_dir, f'xgb_dept_{dept_id}.pkl')
    joblib.dump(model, model_filename)

# Display results
results_df = pd.DataFrame(results).sort_values(by='RMSE')
print("\nGroup-wise XGBoost Performance by Dept:")
print(results_df)



Group-wise XGBoost Performance by Dept:
    Dept  Train Size  Test Size      RMSE      MAE  SMAPE
45    54        3686        922     45.29    36.24  77.39
66    59        4849       1213     95.44    74.07  49.61
65    60        4613       1154    108.99    79.49  30.54
38    28        4898       1225    149.87    98.67  26.30
39    27        4491       1123    392.93   272.82  30.63
..   ...         ...        ...       ...      ...    ...
29    38        5148       1287   9338.91  7295.86  12.68
0      1        5148       1287   9556.01  7219.58  37.54
51    92        5148       1287  10180.12  7221.52  11.04
18     3        5148       1287  10246.95  4378.37  34.77
62    72        4809       1203  11747.11  8275.61  36.62

[70 rows x 6 columns]
